# TC_02: XGBoost clean baseline
Full diagnostic pipeline on clean data using XGB model.
UQ: Deep Ensemble,
XAI: SHAP + LIME

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_xgboost, evaluate_model_rf_xgb
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_sklern
from src.inference_diagnostics.explainability import shap_tab,lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_accuracy_comparison, plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report

config = load_config()
tracker = PerformanceTracker()

### Load data and EDA

In [2]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
report = check_tabular_quality(X_train, y_train, config)
plot_class_distribution(report, "XGB Diabetes Baseline", config)
plot_correlation(report, "XGB Diabetes Baseline", config)
plot_feature_boxplots(X_train, "XGB Diabetes Baseline", config)

Loading dataset.
No missing value found.
Data loaded for 202944 train, 50736 test
 Features: 21
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 18580
Total outlier count: 89295
EDA completed for Diabetes Health Indicators
Saved: results/figures\class_distribution_xgb_diabetes_baseline.png
Saved: results/figures\correlation_xgb_diabetes_baseline.png
Saved: results/figures\boxplot_xgb_diabetes_baseline.png


### Train XGBoost baseline

In [3]:
tracker.start_performance_track()
xgb_model = train_xgboost(X_train, y_train, config)
tracker.stop_performance_track("XGBoost training")

tracker.start_performance_track()
xgb_accuracy, xgb_prediction, xgb_report = evaluate_model_rf_xgb(xgb_model, X_test, y_test)
tracker.stop_performance_track("XGBoost evaluation")

XGBoost training started.
XGBoost training completed.
XGBoost training: Time:0.45s, Memory:143.91MB
Model evaluation started for RF,XGB
{'0': {'precision': 0.8787056453270836, 'recall': 0.9788169555957588, 'f1-score': 0.9260635474330781, 'support': 43667.0}, '1': {'precision': 0.558261700095511, 'recall': 0.16536992502475598, 'f1-score': 0.25515660809778457, 'support': 7069.0}, 'accuracy': 0.8654801324503312, 'macro avg': {'precision': 0.7184836727112973, 'recall': 0.5720934403102573, 'f1-score': 0.5906100777654313, 'support': 50736.0}, 'weighted avg': {'precision': 0.8340584865277698, 'recall': 0.8654801324503312, 'f1-score': 0.8325867034926573, 'support': 50736.0}}
XGBoost evaluation: Time:0.01s, Memory:0.12MB


{'operation': 'XGBoost evaluation', 'time_seconds': 0.01, 'memory_usage': 0.12}

### Uncertainty Quantification (Deep Ensembles)

In [4]:
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_sklern(train_xgboost, X_train, y_train, X_test, config)
tracker.stop_performance_track("XGB Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
de_y_prediction = de_means_probabilities.argmax(axis=1)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "XGB Deep Ensemble", de_threshold, config)

Deep Ensemble started for tabular and sklern.
Training model with seed 0
XGBoost training started.
XGBoost training completed.
Training model with seed 1
XGBoost training started.
XGBoost training completed.
Training model with seed 2
XGBoost training started.
XGBoost training completed.
Training model with seed 3
XGBoost training started.
XGBoost training completed.
Training model with seed 4
XGBoost training started.
XGBoost training completed.
Deep Ensemble finished for tabular sklern.
XGB Deep Ensemble prediction: Time:0.96s, Memory:0.77MB
Deep Ensembl uncertainty status:
Mean: 8.777068849497027e-09
Max: 3.3527612686157227e-08
 90th percentile (threshold): 0.0
Saved: results/figures\uncertainty_distribution_xgb_deep_ensemble.png


### Explainability (SHAP and LIME)

In [5]:
# SHAP
tracker.start_performance_track()
shap_values, shap_samples = shap_tab(xgb_model, X_train, X_test, config, is_pytorch = False)
tracker.stop_performance_track("XGB SHAP")
print(f"SHAP values shape: {shap_values.shape}")

SHAP started.


  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
XGB SHAP: Time:8.48s, Memory:1.92MB
SHAP values shape: (200, 21, 2)


In [6]:
# LIME
tracker.start_performance_track()
lime_explanations, lime_samples = lime_tab(xgb_model, X_train, X_test, config, is_pytorch = False)
tracker.stop_performance_track("XGB LIME")
print(f"LIME explanations: {len(lime_explanations)}")

LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
XGB LIME: Time:1.82s, Memory:11.57MB
LIME explanations: 200


In [7]:
# Calculate consistency
feature_names = list(X_train.columns)
consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)
print(f"Mean consistency: {consistency_scores.mean()}")
print(f"Min consistency: {consistency_scores.min()}")
print(f"Max consistency: {consistency_scores.max()}")


Calculate consistency tabular.
Mean consistency: 0.6499999999999999
Min consistency: 0.4
Max consistency: 1.0


### Flagging

In [8]:
# Real uncertainty value for XGB is near 0 and meaning less ofr flagging system.
xgb_uq_threshold = float(de_uncertainty.max()) + 1.0
xgb_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, xgb_uq_threshold, 0.5)

print("XGB flagging results:")
xgb_flagging_result = evaluate_flags(xgb_flags, xgb_prediction[:len(consistency_scores)], y_test[:len(consistency_scores)])

plot_flag_distribution(xgb_flags, 'XGB Baseline', config)

XGB flagging results:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 26 Accuracy:88.5%
GREEN: Count: 174 Accuracy:87.4%
Flagging system is working
Saved: results/figures\flagging_distribution_xgb_baseline.png


### Save baseline report


In [9]:
xgb_baseline = {
    'model': 'XGBoost',
    'accuracy': round(xgb_accuracy, 4),
    'classification_report': xgb_report,
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold,
        'findings': 'Near zero uncertainty and XGB internal ensemble makes external DE redundant'
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging': xgb_flagging_result,
    'performance': tracker.get_results()
}

report_output = generate_report(config,'Diabetes - XGB clean baseline', stage1_result = xgb_baseline )
save_report(report_output, 'TC_02_XGBoost_Clean_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
